# Python의 Module

> Computational Thinking and SW Coding  
> Department of Computer Science, KNU

---

### 오늘 배울 것

1. **개념** — `import`가 실제로 무엇을 하는가? (namespace, `__name__`)
2. **직접 만들기** — 중간고사 피자 문제를 module로 정리하기
3. **표준 라이브러리** — `math`, `datetime`, `random`을 우리가 만든 것과 같은 시선으로 보기

### 큰 그림

> **Module은 새로 짜는 것이 아니라, 자라난 코드를 정리하는 도구입니다.**

오늘은 시험 문제 하나가 어떻게 자라나서 module화가 *필요해지는지*,
그리고 그것이 결국 표준 라이브러리와 같은 원리라는 것을 함께 봅니다.

---

# Part 1. Module이란 무엇인가?

> "module"이라는 단어를 자주 쓰지만, 정확히 무엇일까요?

## 한 줄 정의

> **Module은 그저 `.py` 파일 하나입니다.**

`math.py`, `random.py`, `datetime.py` … 우리가 `import`하는 모든 것은 결국 누군가 작성한 Python 파일일 뿐입니다.

따라서 **여러분도 module을 만들 수 있습니다.** 오늘 직접 만들어볼 거예요.

## 1.1 `import`는 "선언"이 아니라 "실행"이다

많은 학생들이 `import`를 *함수를 불러오는 선언* 정도로 오해합니다.  
사실은 다릅니다:

> **`import math`는 그 시점에 `math.py` 파일을 위에서 아래로 한 번 실행합니다.**

그리고 그 결과로 만들어진 이름들을 `math`라는 namespace에 묶어둡니다.

### 직접 확인해보기

`math` module을 import해서, 그 안에 어떤 이름들이 들어있는지 들여다봅시다.

In [ ]:
import math

# math 모듈 안에 정의된 이름들 (앞 20개만)
names = [n for n in dir(math) if not n.startswith('_')] # list comprehension
print(f"math 모듈의 이름 개수: {len(names)}")
print(names[:20])

math 모듈의 이름 개수: 62
['acos', 'acosh', 'asin', 'asinh', 'atan', 'atan2', 'atanh', 'cbrt', 'ceil', 'comb', 'copysign', 'cos', 'cosh', 'degrees', 'dist', 'e', 'erf', 'erfc', 'exp', 'exp2']


## 1.2 Module = 이름들의 사전(dict)

Python에서 module은 사실상 dict입니다. 이름과 값의 묶음이죠.  
`math.__dict__`로 직접 까볼 수 있습니다.

In [2]:
# math.__dict__ 의 일부만 보기
items = list(math.__dict__.items())
for name, value in items[:5]:
    print(f"{name!r:20s} -> {value}")

'__name__'           -> math
'__doc__'            -> This module provides access to the mathematical functions
defined by the C standard.
'__package__'        -> 
'__loader__'         -> <class '_frozen_importlib.BuiltinImporter'>
'__spec__'           -> ModuleSpec(name='math', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in')


**왜 이게 중요한가요?**

`math.pi`라고 쓸 때, 우리는 사실 `math`라는 dict에서 `'pi'`라는 키를 꺼내 쓰는 것과 같습니다.  
이게 **namespace**입니다 — 말 그대로 *이름들의 공간*.

In [2]:
# 같은 결과
print(math.pi)
print(math.__dict__['pi'])

3.141592653589793
3.141592653589793


## 1.3 `import X` vs `from X import Y`

단순히 *타이핑이 짧다*의 문제가 아닙니다.  
**어느 namespace에 이름이 들어가는지**가 다릅니다.

In [4]:
import math
# 이 경우 'math'라는 이름만 가져옴
# 사용: math.sqrt(16)
print(math.sqrt(16))

4.0


In [5]:
from math import sqrt
# 이 경우 'sqrt'라는 이름이 직접 들어옴
# 사용: sqrt(16)  (math. 없이!)
print(sqrt(16))

4.0


두 경우의 차이를 좀 더 명확히 보려면 — 현재 namespace에 어떤 이름이 들어왔는지 확인해봅시다.

In [6]:
# 'math'와 'sqrt'가 모두 현재 namespace에 있는지 확인
print('math' in dir())   # True (위에서 import math 했으므로)
print('sqrt' in dir())   # True (위에서 from math import sqrt 했으므로)

True
True


## 1.4 `__name__`의 정체

Python 코드에서 자주 보이는 다음 패턴:

```python
if __name__ == "__main__":
    ...
```

이게 무엇일까요?

> **`__name__`은 "이 파일이 지금 어떤 자격으로 실행되고 있는지"를 담은 변수입니다.**

- 직접 실행되면 → `"__main__"`
- 다른 파일에서 import되면 → 모듈 이름

확인해봅시다.

In [7]:
# 이 노트북을 직접 실행 중이면 __name__은 보통 '__main__' 입니다
print(__name__)

__main__


In [8]:
# math 모듈은 import된 상태이므로, math.__name__ 은 'math'
print(math.__name__)

math


### `if __name__ == "__main__":`의 의미

이 한 줄은 *"이 파일이 라이브러리로 쓰일 때는 실행되지 말아야 할 코드"*를 격리하는 관용구입니다.

오늘 step5에서 직접 만들어볼 때 이 의미가 생생하게 와닿을 거예요.

참고 : https://docs.python.org/3/library/__main__.html

## 1.5 잠깐, 그럼 표준 라이브러리도…?

여러분이 방금 본 `math`, 사실 누군가 작성한 `.py` 파일입니다.  
그리고 **여러분도 똑같이 만들 수 있습니다.**

그럼 만들어볼까요? 마침 좋은 재료가 있습니다 — **중간고사 피자 문제**.

---

# Part 2. 직접 Module 만들기

## 2.0 시험 문제 복기

> Anna는 파티에 친구들을 초대하려고 합니다.  
> 파티 참석 인원(Anna 포함)을 입력 받아, 모든 사람이 최소 1조각씩 먹을 수 있도록  
> 주문해야 할 피자의 최소 개수를 계산하는 Python 코드를 작성하시오.  
> (단, 피자 1판은 8조각으로 나누어집니다.)

가장 간단한 풀이:

```python
people = int(input("파티 참석 인원: "))
pizzas = (people + 7) // 8
print(f"필요한 피자: {pizzas}판")
```

핵심은 `(people + 7) // 8` — **올림 나눗셈** 관용구입니다.

In [9]:
# 함수로 표현해두면 테스트가 편합니다
def calculate_pizzas_v0(people):
    return (people + 7) // 8

# 검증
print(calculate_pizzas_v0(1))   # 1
print(calculate_pizzas_v0(8))   # 1
print(calculate_pizzas_v0(9))   # 2
print(calculate_pizzas_v0(16))  # 2
print(calculate_pizzas_v0(17))  # 3

1
1
2
2
3


### 왜 `+7`일까요?

8조각으로 나누었을 때 나머지가 1이라도 있으면 한 판 더 필요합니다.  
`+7`을 미리 더해두면 — 나누어떨어질 때를 빼고 — 항상 다음 정수로 올라갑니다.

> 나중에 `math.ceil`을 만나면 "아, 같은 일을 하는구나"를 깨닫게 됩니다. 잠시 기억해두세요.

## 2.1 확장 1단계: 1인당 조각 수

> "Anna의 친구들이 식욕이 좋아서 1인당 2조각, 3조각씩 먹고 싶어 합니다."

함수를 살짝 확장해봅시다.

In [3]:
def calculate_pizzas(people, slices_per_person=1):
    """필요한 피자 판수를 계산한다."""
    total_slices = people * slices_per_person
    return (total_slices + 7) // 8

# 테스트
print(calculate_pizzas(5))         # 1 (5조각 → 1판)
print(calculate_pizzas(5, 2))      # 2 (10조각 → 2판)
print(calculate_pizzas(17, 2))     # 5 (34조각 → 5판)

1
2
5


## 2.2 확장 2단계: 메뉴와 가격

> "노모어, 잭슨, 도미노, 파파존스 중에서 시키려는데 가격이 다 다릅니다."

> ⚠ 가격은 예시이며 실제와 다를 수 있습니다.

In [11]:
MENU = {
    "노모어피자": 12000,
    "잭슨피자": 15000,
    "도미노피자": 25000,
    "파파존스": 28000,
}

def show_menu():
    print("=== 메뉴 ===")
    for name, price in MENU.items():
        print(f"  {name}: {price:,}원")

show_menu()

=== 메뉴 ===
  노모어피자: 12,000원
  잭슨피자: 15,000원
  도미노피자: 25,000원
  파파존스: 28,000원


## 2.3 확장 3단계: 여러 종류 주문

> "한 종류만 시키는 게 아니라, 여러 종류를 섞어서 주문하고 싶다."

이제 *주문 목록*이라는 자료구조가 필요합니다.  
보통 dict의 list로 표현합니다.

In [5]:
# 주문 예시
sample_order = [
    {"name": "노모어피자", "count": 2},
    {"name": "도미노피자", "count": 1},
    {"name": "파파존스",   "count": 1},
]

MENU = {
    "노모어피자": 12000,
    "잭슨피자": 15000,
    "도미노피자": 25000,
    "파파존스": 28000,
}

def calculate_total(order):
    """주문 목록의 총 금액을 계산한다."""
    total = 0
    for item in order:
        total += MENU[item["name"]] * item["count"]
    return total

def calculate_total_pizzas(order):
    """주문 목록의 총 판수를 합한다."""
    return sum(item["count"] for item in order)

print(f"총 판수: {calculate_total_pizzas(sample_order)}")
print(f"총 금액: {calculate_total(sample_order):,}원")

총 판수: 4
총 금액: 77,000원


### ⚠ 헷갈리기 쉬운 두 함수

- `calculate_pizzas(people, slices)` — 사람 수로부터 **필요한** 판수를 구함 (시험 문제의 핵심)
- `calculate_total_pizzas(order)` — 주문 목록의 **주문된** 판수를 합함

이름이 비슷해도 역할이 완전히 다릅니다.  
이게 나중에 module 분리할 때 **서로 다른 책임**이라는 점이 명확해집니다.

## 2.4 확장 4단계: 영수증 출력

In [9]:
def print_receipt(order, total):
    print("=" * 35)
    print("           주문 영수증")
    print("=" * 35)
    for item in order:
        name = item["name"]
        count = item["count"]
        subtotal = MENU[name] * count
        print(f"  {name:10s} {count}판  {subtotal:>10,}원")
    print("-" * 35)
    print(f"  합계: {total:>22,}원")
    print("=" * 35)

print_receipt(sample_order, calculate_total(sample_order))

           주문 영수증
  노모어피자      2판      24,000원
  도미노피자      1판      25,000원
  파파존스       1판      28,000원
-----------------------------------
  합계:                 77,000원


## 2.5 잠깐 — 코드를 둘러봅시다

지금까지 우리는:
- `MENU` 데이터 1개
- 함수 5개 (`show_menu`, `calculate_pizzas`, `calculate_total`, `calculate_total_pizzas`, `print_receipt`)
- 메인 흐름

을 한 곳에 만들었습니다. 만약 이걸 한 `.py` 파일에 다 담으면 70~80줄짜리가 됩니다.

### 질문

1. 화면에 한눈에 들어오나요?
2. `print_receipt()`만 다른 프로젝트에서 쓰고 싶다면?
3. `MENU`만 따로 관리하고 싶다면? (예: 가격이 자주 바뀐다면)
4. 누군가 `calculate_pizzas()`의 버그를 고치려면 어디부터 봐야 하나요?

> 이 질문들이 **module 분리의 출발점**입니다.

---

# Part 3. Module로 분리하기

## 3.1 어떻게 나눌까?

**책임이 다른 코드는 다른 파일로** — 이게 module 분리의 첫 번째 원칙입니다.

```
project/
├── pizza_menu.py      # 메뉴 데이터 + show_menu()
├── pizza_calc.py      # 계산 함수들
├── pizza_order.py     # 주문 받기 (입력)
├── pizza_receipt.py   # 영수증 출력
└── main.py            # 전체 흐름 조립
```

각 파일의 내용을 살펴봅시다 (자세한 코드는 zip 실습자료의 `step5_모듈분리/`에 있습니다).

### `pizza_menu.py` — 데이터와 메뉴 출력

```python
MENU = {
    "노모어피자": 12000,
    "잭슨피자": 15000,
    "도미노피자": 25000,
    "파파존스": 28000,
}

def show_menu():
    print("=== 메뉴 ===")
    for name, price in MENU.items():
        print(f"  {name}: {price:,}원")

if __name__ == "__main__":
    print("== pizza_menu 자체 테스트 ==")
    show_menu()
```

### `pizza_calc.py` — 계산 (다른 모듈에서 데이터를 가져옴!)

```python
from pizza_menu import MENU   # ← 핵심!

def calculate_pizzas(people, slices_per_person=1):
    total_slices = people * slices_per_person
    return (total_slices + 7) // 8

def calculate_total(order):
    total = 0
    for item in order:
        total += MENU[item["name"]] * item["count"]
    return total

def calculate_total_pizzas(order):
    return sum(item["count"] for item in order)
```

> **여기서 Part 1의 개념이 살아납니다.**  
> `from pizza_menu import MENU`는 — `pizza_menu.py`를 한 번 실행하고, 그 결과 만들어진 `MENU`라는 이름을 가져옵니다.  
> `MENU`도 namespace에 든 이름이니까 가져올 수 있는 거죠.

### `main.py` — 전체 흐름 조립

```python
from pizza_menu import show_menu
from pizza_calc import calculate_pizzas, calculate_total, calculate_total_pizzas
from pizza_order import take_order
from pizza_receipt import print_receipt

def main():
    show_menu()
    people = int(input("파티 참석 인원: "))
    slices_per_person = int(input("1인당 조각 수: "))
    needed = calculate_pizzas(people, slices_per_person)
    print(f"최소 {needed}판이 필요합니다.")

    order = take_order()
    print_receipt(order, calculate_total(order))

if __name__ == "__main__":
    main()
```

`main.py`는 **지휘자**입니다. 직접 일은 하지 않고, 각 모듈에 시킬 뿐입니다.

## 3.2 `if __name__ == "__main__":`이 진가를 발휘하는 순간

각 모듈 하단에 자체 테스트 코드를 두면:

- `python pizza_calc.py` → 자체 테스트가 실행됨 (`__name__ == "__main__"`)
- `from pizza_calc import ...` → 테스트는 무시되고 함수만 import됨 (`__name__ == "pizza_calc"`)

> **같은 파일이 두 가지 자격으로 동작합니다.**

이게 Part 1.4에서 다룬 `__name__`의 실용적 의미입니다.

## 3.3 Module 분리의 이득

| 분리 전 | 분리 후 |
|--|--|
| 한 파일 70~80줄 | 각 파일 20~30줄 |
| 모든 코드를 다 봐야 한 곳 수정 | 책임 영역만 보면 됨 |
| 다른 프로젝트에서 재사용 어려움 | `pizza_calc`만 가져다 쓸 수 있음 |
| 누가 누구에 의존하는지 불분명 | `import` 줄로 명시적 |

> 핵심: **module은 코드가 자라난 결과 자연스럽게 도입되는 것**이지, 처음부터 작은 코드를 억지로 쪼개는 게 아닙니다.

---

# Part 4. 표준 라이브러리로 넘어가기

## 4.1 우리가 만든 것 == 표준 라이브러리

> `pizza_menu.py`도, `math.py`도, 둘 다 `.py` 파일입니다.  
> 차이는 단지: 표준 라이브러리는 Python을 설치할 때 함께 들어있다는 것.

같은 시선으로 봐봅시다.

## 4.2 `math.ceil` — 우리가 손으로 짠 그것

기억하시죠? 시험 문제에서:

```python
pizzas = (people + 7) // 8   # 올림 나눗셈
```

표준 라이브러리에는 이미 이 일을 하는 함수가 있습니다.

In [10]:
import math

# 우리 방식
def my_way(people):
    return (people + 7) // 8

# 표준 라이브러리 방식
def std_way(people):
    return math.ceil(people / 8)

# 비교
for n in [1, 8, 9, 16, 17, 25]:
    print(f"{n}명: 우리={my_way(n)}, 표준={std_way(n)} → 같음? {my_way(n) == std_way(n)}")

1명: 우리=1, 표준=1 → 같음? True
8명: 우리=1, 표준=1 → 같음? True
9명: 우리=2, 표준=2 → 같음? True
16명: 우리=2, 표준=2 → 같음? True
17명: 우리=3, 표준=3 → 같음? True
25명: 우리=4, 표준=4 → 같음? True


**어느 쪽이 좋을까요?**

- `(people + 7) // 8` — 트릭이라 *왜 +7인지* 한 번 생각해야 함
- `math.ceil(people / 8)` — *"올림"* 이라는 의도가 코드에 그대로 드러남

가독성 측면에선 후자가 보통 낫습니다. 다만 트릭을 이해하는 것도 중요한 학습이에요.

## 4.3 `datetime` — 영수증에 시각 찍기

In [11]:
import datetime

now = datetime.datetime.now()
print(f"현재 시각: {now}")
print(f"포맷팅:    {now:%Y-%m-%d %H:%M}")
print(f"날짜만:    {now:%Y년 %m월 %d일}")

현재 시각: 2026-04-30 19:04:17.661854
포맷팅:    2026-04-30 19:04
날짜만:    2026년 04월 30일


## 4.4 `random` — 영수증 번호 생성

In [12]:
import random

receipt_no = random.randint(1000, 9999)
print(f"영수증 번호: #{receipt_no}")

# 매번 다르게 나옴
for _ in range(3):
    print(f"  #{random.randint(1000, 9999)}")

영수증 번호: #4965
  #2472
  #3124
  #8742


## 4.5 영수증을 한 단계 업그레이드

In [13]:
import datetime
import random

def print_receipt_v2(order, total):
    receipt_no = random.randint(1000, 9999)
    now = datetime.datetime.now()

    print("=" * 35)
    print("           주문 영수증")
    print(f"  영수증 번호: #{receipt_no}")
    print(f"  주문 시각:   {now:%Y-%m-%d %H:%M}")
    print("=" * 35)
    for item in order:
        name = item["name"]
        count = item["count"]
        subtotal = MENU[name] * count
        print(f"  {name:10s} {count}판  {subtotal:>10,}원")
    print("-" * 35)
    print(f"  합계: {total:>22,}원")
    print("=" * 35)

print_receipt_v2(sample_order, calculate_total(sample_order))

           주문 영수증
  영수증 번호: #2368
  주문 시각:   2026-04-30 19:04
  노모어피자      2판      24,000원
  도미노피자      1판      25,000원
  파파존스       1판      28,000원
-----------------------------------
  합계:                 77,000원


---

# 정리

## 오늘 배운 것

1. **`import`는 실행이다** — `.py` 파일을 한 번 돌리고, 그 결과 namespace를 만든다.
2. **Module = `.py` 파일** — 표준 라이브러리도, 우리가 만든 것도 같은 원리.
3. **Namespace = 이름의 공간** — `dir()`, `__dict__`로 들여다볼 수 있다.
4. **`__name__`** — 파일이 어떤 자격으로 실행 중인지 알려주는 변수.
5. **Module화는 자라난 코드를 정리하는 도구** — 처음부터 쪼개는 게 아님.
6. **표준 라이브러리는 거인의 어깨** — 우리가 손으로 짤 수도 있는 것을, 검증된 코드로 즉시 쓸 수 있게 해줌.

## 실습 과제

zip 실습자료의 `step5_모듈분리/`를 받아서:

1. 각 module을 단독 실행해보기 (`python pizza_menu.py` 등) → `if __name__ == "__main__":`이 어떻게 동작하는지 관찰
2. `python main.py`를 실행해서 전체 흐름 확인
3. **자유 확장**: `pizza_calc.py`에 새 함수를 추가하기 (예: 1인당 평균 가격 계산)

## 다음 강의 예고

- 직접 만든 `pizza_*` module들을 *package*로 묶기 (디렉토리 + `__init__.py`)
- `pip`으로 외부 라이브러리 설치하기 — *"누군가 만든 module을 가져다 쓰는" 또 하나의 방법*

---

> 질문은 언제든 Slack으로! 